# 单谱宇宙射线清理对照

这个 notebook 只看单谱宇宙射线清理本身，不做 AsLS、不裁切、不插值、不做 group 兜底

展示内容：

1. 原始谱 vs narrow cleaned
2. narrow cleaned vs narrow + peak-morph cleaned
3. narrow + peak-morph cleaned vs final cleaned
4. residual 阶段处理区域局部放大

默认从 `DATASET_NAME` 对应数据集的 `init` 目录里随机抽取一条原始谱。如果要固定单条样本，填 `SAMPLE_PATH`；如果要固定某个文件夹内随机抽样，填 `SAMPLE_FOLDER`


In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.signal import find_peaks, peak_prominences, peak_widths


def find_project_root(start=None):
    """从当前目录向上查找项目根目录，兼容从 notebooks/ 里启动"""
    current = Path(start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "raman").is_dir() and (path / "dataset").is_dir():
            return path
    return current


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "raman" or name.startswith("raman."):
        del sys.modules[name]

from raman.data.build import COMMON_BAD_BANDS, DEFAULT_PIPELINE_CONFIG
from raman.data.offline import (
    _estimate_noise_from_diff,
    _median_filter_1d,
    _merge_intervals,
    _replace_segment,
    _residual_z_score,
    remove_cosmic_rays,
)
from raman.data.profiles import get_dataset_dir, get_profile
from raman.data.spectrum import build_valid_mask, read_arc_data

sns.set_theme(style="whitegrid", context="talk", font="Microsoft YaHei")
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "Microsoft YaHei"
plt.rcParams["figure.figsize"] = (11, 5)

DATASET_NAME = "细菌"
SAMPLE_PATH = "dataset/细菌/init/Proteus/PVU01/cell7_Area01_000_shift.arc_data"  # 填文件路径时固定单条样本；None 时走随机抽样
SAMPLE_FOLDER = None  # 填相对文件夹时只在该文件夹内随机抽样；None 时用 init
SAMPLE_SEED = None  # 设为整数可复现随机抽样

profile = get_profile(DATASET_NAME)
dataset_dir = get_dataset_dir(profile, PROJECT_ROOT)
cfg = DEFAULT_PIPELINE_CONFIG
BAD_BANDS = COMMON_BAD_BANDS


def resolve_sample_path():
    if SAMPLE_PATH:
        path = Path(SAMPLE_PATH)
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.is_file():
            raise FileNotFoundError(f"样本不存在：{path}")
        return path

    if SAMPLE_FOLDER:
        root = Path(SAMPLE_FOLDER)
        if not root.is_absolute():
            root = dataset_dir / root
    else:
        root = dataset_dir / profile.root_init

    matches = sorted(root.rglob("*.arc_data"))
    if not matches:
        raise FileNotFoundError(f"未找到 .arc_data：{root}")
    return random.Random(SAMPLE_SEED).choice(matches)


def add_bad_band_spans(ax, bad_bands, label="bad bands"):
    if getattr(ax, "_bad_band_spans_added", False):
        return
    for idx, (band_min, band_max) in enumerate(bad_bands):
        ax.axvspan(
            band_min,
            band_max,
            color="gray",
            alpha=0.15,
            label=label if idx == 0 else None,
        )
    ax._bad_band_spans_added = True


def plot_without_bad_bands(ax, wn, sp, bad_bands, **kwargs):
    """坏段位置断线，只保留灰色 mask"""
    sp_plot = np.asarray(sp, dtype=np.float32).copy()
    keep_mask = build_valid_mask(np.asarray(wn), bad_bands)
    if keep_mask is not None:
        sp_plot[~keep_mask] = np.nan
    return ax.plot(wn, sp_plot, **kwargs)


def style_axis(ax, title, ylabel="Intensity"):
    add_bad_band_spans(ax, BAD_BANDS)
    ax.set_title(title)
    ax.set_xlabel("Wavenumber (cm$^{-1}$)")
    ax.set_ylabel(ylabel)
    ax.legend(loc="best")

In [ ]:
def run_cosmic_cleanup_trace(wn, sp, valid_mask, cfg):
    """按库内清理逻辑执行 narrow + peak morphology，并记录被替换的位置"""
    raw = np.asarray(sp, dtype=np.float32)
    valid_mask = np.asarray(valid_mask, dtype=bool)

    diffs = np.diff(wn)
    diffs = np.abs(diffs[np.isfinite(diffs) & (np.abs(diffs) > 1e-8)])
    step = max(float(np.median(diffs)), 1e-6) if diffs.size else 1.0
    narrow_window = max(3, int(round(float(cfg.cosmic_ray_narrow_window_cm) / step)))
    if narrow_window % 2 == 0:
        narrow_window += 1
    peak_pad_points = max(0, int(round(float(cfg.cosmic_ray_peak_pad_cm) / step)))

    narrow_clean = raw.copy()
    narrow_mask = np.zeros(raw.shape, dtype=bool)
    narrow_iter_masks = []

    for _ in range(int(cfg.cosmic_ray_max_iter)):
        local_median = _median_filter_1d(narrow_clean, narrow_window)
        residual = narrow_clean - local_median
        z_score = _residual_z_score(residual, valid_mask)
        if z_score is None:
            break

        spike_mask = valid_mask & (z_score > float(cfg.cosmic_ray_threshold))
        if not spike_mask.any():
            break

        narrow_iter_masks.append(spike_mask.copy())
        narrow_mask |= spike_mask
        narrow_clean[spike_mask] = local_median[spike_mask]

    peak_clean = narrow_clean.copy()
    peak_mask = np.zeros(raw.shape, dtype=bool)
    peak_segments = []
    noise = _estimate_noise_from_diff(peak_clean, valid_mask)

    peaks, _ = find_peaks(peak_clean)
    peaks = peaks[valid_mask[peaks]]
    if noise > 1e-8 and peaks.size > 0:
        prominences = peak_prominences(peak_clean, peaks)[0]
        widths, _, left_ips, right_ips = peak_widths(
            peak_clean,
            peaks,
            rel_height=float(cfg.cosmic_ray_peak_rel_height),
        )
        width_cm = widths * step
        prominence_z = prominences / noise
        ratio = prominence_z / np.maximum(width_cm, step)
        selected = (
            (prominence_z >= float(cfg.cosmic_ray_peak_prominence_z))
            & (width_cm <= float(cfg.cosmic_ray_peak_width_max_cm))
            & (ratio >= float(cfg.cosmic_ray_peak_ratio_z_per_cm))
        )

        fallback = _median_filter_1d(peak_clean, 7)
        candidates = []
        for peak, left_ip, right_ip, prom_z, width, ratio_value in zip(
            peaks[selected],
            left_ips[selected],
            right_ips[selected],
            prominence_z[selected],
            width_cm[selected],
            ratio[selected],
        ):
            start = max(0, int(np.floor(left_ip)) - peak_pad_points)
            end = min(peak_clean.size, int(np.ceil(right_ip)) + peak_pad_points + 1)
            if start < end:
                candidates.append(
                    {
                        "start": start,
                        "end": end,
                        "peak": int(peak),
                        "width_cm": float(width),
                        "prominence_z": float(prom_z),
                        "ratio": float(ratio_value),
                    }
                )

        for start, end in _merge_intervals((item["start"], item["end"]) for item in candidates):
            segment_mask = np.zeros(peak_clean.shape, dtype=bool)
            segment_mask[start:end] = True
            segment_mask &= valid_mask
            if not segment_mask.any():
                continue

            start_idx = int(np.where(segment_mask)[0][0])
            end_idx = int(np.where(segment_mask)[0][-1]) + 1
            before = peak_clean[start_idx:end_idx].copy()
            _replace_segment(peak_clean, fallback, start_idx, end_idx, valid_mask)
            peak_mask[start_idx:end_idx] = True
            related = [
                item for item in candidates
                if item["start"] < end_idx and item["end"] > start_idx
            ]
            peak_segments.append(
                {
                    "start": start_idx,
                    "end": end_idx,
                    "wn_min": float(wn[start_idx]),
                    "wn_max": float(wn[end_idx - 1]),
                    "width_points": int(end_idx - start_idx),
                    "width_cm": float(max(item["width_cm"] for item in related)),
                    "prominence_z": float(max(item["prominence_z"] for item in related)),
                    "ratio": float(max(item["ratio"] for item in related)),
                    "delta_mean": float(np.mean(before - peak_clean[start_idx:end_idx])),
                }
            )

    return {
        "raw": raw,
        "narrow_clean": narrow_clean,
        "peak_clean": peak_clean,
        "narrow_mask": narrow_mask,
        "narrow_iter_masks": narrow_iter_masks,
        "peak_mask": peak_mask,
        "peak_segments": peak_segments,
        "noise": noise,
    }


sample_path = resolve_sample_path()
wn, sp = read_arc_data(sample_path)
if wn.size == 0 or sp.size == 0:
    raise ValueError(f"读取失败：{sample_path}")

cosmic_valid_mask = np.ones_like(wn, dtype=bool)

trace = run_cosmic_cleanup_trace(wn, sp, cosmic_valid_mask, cfg)
final_clean, final_stats = remove_cosmic_rays(
    sp,
    window_cm=cfg.cosmic_ray_narrow_window_cm,
    threshold=cfg.cosmic_ray_threshold,
    max_iter=cfg.cosmic_ray_max_iter,
    valid_mask=cosmic_valid_mask,
    wn=wn,
    peak_prominence_z=cfg.cosmic_ray_peak_prominence_z,
    peak_width_max_cm=cfg.cosmic_ray_peak_width_max_cm,
    peak_ratio_z_per_cm=cfg.cosmic_ray_peak_ratio_z_per_cm,
    peak_pad_cm=cfg.cosmic_ray_peak_pad_cm,
    peak_rel_height=cfg.cosmic_ray_peak_rel_height,
    residual_threshold_z=cfg.cosmic_ray_residual_threshold_z,
    residual_max_points_fraction=cfg.cosmic_ray_residual_max_points_fraction,
)
trace["final_clean"] = final_clean
trace["final_stats"] = final_stats
trace["residual_mask"] = np.abs(final_clean - trace["peak_clean"]) > 1e-6

print(f"project_root = {PROJECT_ROOT}")
print(f"dataset_dir = {dataset_dir}")
print(f"sample_path = {sample_path}")
print(f"bad_bands = {BAD_BANDS}")
print(
    f"narrow: window_cm={cfg.cosmic_ray_narrow_window_cm}, "
    f"threshold={cfg.cosmic_ray_threshold}, max_iter={cfg.cosmic_ray_max_iter}"
)
print(
    "peak morphology: "
    f"prominence_z>={cfg.cosmic_ray_peak_prominence_z}, "
    f"width_cm<={cfg.cosmic_ray_peak_width_max_cm}, "
    f"ratio>={cfg.cosmic_ray_peak_ratio_z_per_cm}, "
    f"pad_cm={cfg.cosmic_ray_peak_pad_cm}"
)
print(f"noise scale = {trace['noise']:.6f}")
print(f"narrow 替换点数: {int(trace['narrow_mask'].sum())}")
print(f"peak morphology 替换点数: {int(trace['peak_mask'].sum())}")
print(f"peak morphology 片段数: {len(trace['peak_segments'])}")
print(f"residual 平台残留替换点数: {final_stats.residual}")
print(f"final 总替换点数: {final_stats.total}")

In [ ]:
# 1. 原始谱和 narrow 结果
fig, ax = plt.subplots(figsize=(12, 6))
plot_without_bad_bands(ax, wn, trace["raw"], BAD_BANDS, label="raw", alpha=0.45, linewidth=1.0)
plot_without_bad_bands(ax, wn, trace["narrow_clean"], BAD_BANDS, label="narrow cleaned", alpha=0.95, linewidth=1.2)
style_axis(ax, f"1. 原始谱 / narrow cleaned | narrow={int(trace['narrow_mask'].sum())} 点")
plt.show()

In [ ]:
# 2. narrow 结果和 narrow + peak 结果
fig, ax = plt.subplots(figsize=(12, 6))
plot_without_bad_bands(ax, wn, trace["narrow_clean"], BAD_BANDS, label="narrow cleaned", alpha=0.65, linewidth=1.0)
plot_without_bad_bands(ax, wn, trace["peak_clean"], BAD_BANDS, label="narrow + peak-morph cleaned", alpha=0.95, linewidth=1.2)

peak_mask = trace["peak_mask"]
visible_mask = peak_mask & build_valid_mask(wn, BAD_BANDS)
if visible_mask.any():
    ax.scatter(
        wn[visible_mask],
        trace["narrow_clean"][visible_mask],
        s=24,
        color="darkorange",
        label="peak-morph original points",
        zorder=5,
    )
else:
    ax.text(0.02, 0.92, "当前样本没有可见的 peak-morph 替换点", transform=ax.transAxes)

style_axis(ax, f"2. narrow cleaned / narrow + peak-morph cleaned | peak={int(peak_mask.sum())} 点")
plt.show()

In [ ]:
# 3. narrow + peak 结果和最终结果
fig, ax = plt.subplots(figsize=(12, 6))
plot_without_bad_bands(ax, wn, trace["peak_clean"], BAD_BANDS, label="narrow + peak-morph cleaned", alpha=0.65, linewidth=1.0)
plot_without_bad_bands(ax, wn, trace["final_clean"], BAD_BANDS, label="final cleaned", alpha=0.95, linewidth=1.2)

residual_mask = trace["residual_mask"]
visible_mask = residual_mask & build_valid_mask(wn, BAD_BANDS)
if visible_mask.any():
    ax.scatter(
        wn[visible_mask],
        trace["peak_clean"][visible_mask],
        s=22,
        color="crimson",
        label="residual original points",
        zorder=5,
    )
else:
    ax.text(0.02, 0.92, "当前样本没有可见的 residual 替换点", transform=ax.transAxes)

style_axis(ax, f"3. narrow + peak-morph cleaned / final cleaned | residual={int(residual_mask.sum())} 点")
plt.show()

In [ ]:
# 4. residual 阶段处理区域局部放大，默认最多展示前 6 段
residual_mask = np.asarray(trace["residual_mask"], dtype=bool)
visible_residual_mask = residual_mask & build_valid_mask(wn, BAD_BANDS)
edges = np.diff(np.r_[False, visible_residual_mask, False].astype(np.int8))
segments = list(zip(np.where(edges == 1)[0], np.where(edges == -1)[0]))[:6]

if not segments:
    print("当前样本没有可见的 residual 阶段处理区域可放大")
else:
    fig, axes = plt.subplots(len(segments), 1, figsize=(12, 3.2 * len(segments)), squeeze=False)
    axes = axes[:, 0]
    for idx, (ax, (start, end)) in enumerate(zip(axes, segments), start=1):
        wn_min = float(wn[start])
        wn_max = float(wn[end - 1])
        pad = max((wn_max - wn_min) * 4, 20.0)
        left = wn_min - pad
        right = wn_max + pad
        local = (wn >= left) & (wn <= right)

        plot_without_bad_bands(ax, wn[local], trace["peak_clean"][local], BAD_BANDS, label="before residual (narrow + peak)", alpha=0.75, linewidth=1.1)
        plot_without_bad_bands(ax, wn[local], trace["final_clean"][local], BAD_BANDS, label="after residual (final)", alpha=0.95, linewidth=1.2)
        ax.scatter(
            wn[start:end],
            trace["peak_clean"][start:end],
            s=24,
            color="crimson",
            label="residual original points",
            zorder=5,
        )
        ax.axvspan(wn_min, wn_max, color="crimson", alpha=0.16)
        add_bad_band_spans(ax, BAD_BANDS)
        ax.set_title(f"residual {idx}: {wn_min:.2f}-{wn_max:.2f} cm$^{{-1}}$ | points={end - start}")
        ax.set_ylabel("Intensity")
        ax.legend(loc="best")

    axes[-1].set_xlabel("Wavenumber (cm$^{-1}$)")
    plt.tight_layout()
    plt.show()